In [1]:
import os
import glob
import subprocess
import re
from concurrent.futures import ThreadPoolExecutor

# ================= CONFIGURATION =================
# 你的原始 ADNI DICOM 根目录
INPUT_DIR = "../autodl-tmp/ADNI_raw_data"  
# 转换后的 NIfTI 文件保存目录
OUTPUT_DIR = "../autodl-tmp/NIfTI_output"  
# 线程数（根据你的服务器 CPU 核心数调整，ADNI 数据量大，多线程能快很多）
NUM_THREADS = 4  
# ==========================================

def process_dicom_folder(dcm_folder):
    """
    处理单个包含 .dcm 的最底层文件夹
    """
    # 获取该文件夹下所有的 dcm 文件
    dcm_files = glob.glob(os.path.join(dcm_folder, "*.dcm"))
    if not dcm_files:
        return
    
    # 随便取第一个文件来解析名称
    sample_file = os.path.basename(dcm_files[0])
    
    # 使用正则表达式提取 Subject ID (例如: 010_S_0420) 和 Image ID (例如: I193085)
    # ADNI 的 Subject ID 格式通常为: 3位数字_S_4位数字
    subject_match = re.search(r'\d{3}_S_\d{4}', sample_file)
    # ADNI 的 Image ID 格式通常为: I + 多位数字
    image_id_match = re.search(r'I\d{4,8}', sample_file)
    
    if subject_match and image_id_match:
        subject_id = subject_match.group()
        image_id = image_id_match.group()
        
        # 构造输出文件名: SubjectID_ImageID (不带后缀，dcm2niix会自动加 .nii.gz)
        # 例如: 010_S_0420_I193085
        output_filename = f"{subject_id}_{image_id}"
        
        # 构造 dcm2niix 命令
        # -z y: 压缩为 .nii.gz
        # -f: 指定输出文件名
        # -o: 输出目录
        cmd =[
            "dcm2niix",
            "-z", "y",
            "-m", "y",
            "-ba", "y",
            "-f", output_filename,
            "-o", OUTPUT_DIR,
            dcm_folder
        ]
        
        try:
            # 执行命令，隐藏原本庞大的控制台输出
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            # subprocess.run(cmd, check=True)
            print(f"[成功] {subject_id} - {image_id} 转换完成")
        except subprocess.CalledProcessError:
            print(f"[错误] dcm2niix 转换失败: {dcm_folder}")
    else:
        print(f"[跳过] 无法从文件名解析 ID: {sample_file}")

def main():
    # 确保输出目录存在
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print("正在扫描目录，寻找包含 .dcm 的最底层文件夹...")
    dcm_folders = set()
    
    # os.walk 递归遍历所有目录
    for root, dirs, files in os.walk(INPUT_DIR):
        for file in files:
            if file.endswith(".dcm"):
                dcm_folders.add(root)
                break # 找到一个dcm就说明这个文件夹需要处理，跳出当前文件的循环
                
    dcm_folders = list(dcm_folders)
    print(f"共发现 {len(dcm_folders)} 个待处理的扫描序列文件夹。")
    print(f"开始使用 {NUM_THREADS} 个线程进行批量转换...\n")
    
    # 使用线程池并发执行，大幅缩短时间
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        executor.map(process_dicom_folder, dcm_folders)
        
    print("\n全部转换任务执行完毕！")

if __name__ == "__main__":
    main()

正在扫描目录，寻找包含 .dcm 的最底层文件夹...
共发现 760 个待处理的扫描序列文件夹。
开始使用 4 个线程进行批量转换...

Chris Rorden's dcm2niiX version v1.0.20211006  (JP2:OpenJPEG) GCC11.2.0 x86-64 (64-bit Linux)
Found 160 DICOM file(s)
Convert 160 DICOM as ../autodl-tmp/NIfTI_output/023_S_0083_I35430 (192x192x160x1)
Chris Rorden's dcm2niiX version v1.0.20211006  (JP2:OpenJPEG) GCC11.2.0 x86-64 (64-bit Linux)
Found 166 DICOM file(s)
Convert 166 DICOM as ../autodl-tmp/NIfTI_output/131_S_0457_I109772 (256x256x166x1)
Chris Rorden's dcm2niiX version v1.0.20211006  (JP2:OpenJPEG) GCC11.2.0 x86-64 (64-bit Linux)
Found 166 DICOM file(s)
Convert 166 DICOM as ../autodl-tmp/NIfTI_output/033_S_1098_I129468 (256x256x166x1)
Chris Rorden's dcm2niiX version v1.0.20211006  (JP2:OpenJPEG) GCC11.2.0 x86-64 (64-bit Linux)
Found 166 DICOM file(s)
Convert 166 DICOM as ../autodl-tmp/NIfTI_output/002_S_0685_I37162 (256x256x166x1)
Compress: "/usr/bin/pigz" -b 960 -n -f -6 "../autodl-tmp/NIfTI_output/023_S_0083_I35430.nii"
Conversion required 0.993682 secon

Error: Missing images. Found 32 images, expected 166 slices per volume and instance number (0020,0013) ranges from 1 to 63


Chris Rorden's dcm2niiX version v1.0.20211006  (JP2:OpenJPEG) GCC11.2.0 x86-64 (64-bit Linux)
Found 180 DICOM file(s)
Convert 180 DICOM as ../autodl-tmp/NIfTI_output/021_S_0753_I20169 (256x256x180x1)
Compress: "/usr/bin/pigz" -b 960 -n -f -6 "../autodl-tmp/NIfTI_output/027_S_0403_I28043.nii"
Conversion required 1.297481 seconds (0.352624 for core code).
[成功] 027_S_0403 - I28043 转换完成
Compress: "/usr/bin/pigz" -b 960 -n -f -6 "../autodl-tmp/NIfTI_output/005_S_0610_I17303.nii"
Conversion required 0.880628 seconds (0.249188 for core code).
[成功] 005_S_0610 - I17303 转换完成
Compress: "/usr/bin/pigz" -b 960 -n -f -6 "../autodl-tmp/NIfTI_output/133_S_0488_I59843.nii"
Compress: "/usr/bin/pigz" -b 960 -n -f -6 "../autodl-tmp/NIfTI_output/133_S_0488_I59843_Eq_1.nii"
Conversion required 0.731620 seconds (0.066810 for core code).
[成功] 133_S_0488 - I59843 转换完成
Chris Rorden's dcm2niiX version v1.0.20211006  (JP2:OpenJPEG) GCC11.2.0 x86-64 (64-bit Linux)
Found 180 DICOM file(s)
Convert 180 DICOM as ../au

In [2]:
# import os
# import subprocess
# import re
# from concurrent.futures import ThreadPoolExecutor

# # ================= 配置区 =================
# # 你的原始 ADNI DICOM 根目录
# # 【注意】请确认你的解压路径，有时候解压后会多一层，变成 /autodl-tmp/ADNI_raw_data/ADNI
# INPUT_DIR = "../autodl-tmp/ADNI_raw_data"  
# OUTPUT_DIR = "../autodl-tmp/NIfTI_output"  
# NUM_THREADS = 8  
# # ==========================================

# def process_dicom_folder(dcm_folder):
#     """
#     处理单个最底层文件夹
#     """
#     # 获取文件夹下的所有文件（排除子文件夹）
#     files_in_dir =[f for f in os.listdir(dcm_folder) if os.path.isfile(os.path.join(dcm_folder, f))]
    
#     # 过滤掉常见的 ADNI 附属说明文件，剩下的通常就是 DICOM 图像本身（不管它有没有后缀）
#     valid_files =[f for f in files_in_dir if not f.lower().endswith(('.xml', '.csv', '.txt', '.json', '.pdf'))]
    
#     if not valid_files:
#         return
    
#     # 随便取第一个有效文件的【完整绝对路径】来解析 ID
#     sample_file_path = os.path.join(dcm_folder, valid_files[0])
    
#     # 【核心升级】在“完整路径”中寻找 Subject ID 和 Image ID
#     # 哪怕文件名叫 1-01.DCM，只要它的父文件夹叫 I193085，就能精准抓取！
#     subject_match = re.search(r'\d{3}_S_\d{4}', sample_file_path)
#     image_id_match = re.search(r'I\d{4,8}', sample_file_path)
    
#     if subject_match and image_id_match:
#         subject_id = subject_match.group()
#         image_id = image_id_match.group()
        
#         output_filename = f"{subject_id}_{image_id}"
        
#         cmd =[
#             "dcm2niix",
#             "-z", "y",
#             "-f", output_filename,
#             "-o", OUTPUT_DIR,
#             dcm_folder  # 直接把文件夹路径丢给 dcm2niix，它极其聪明，能自动读取里面的所有切片
#         ]
        
#         try:
#             # 运行 dcm2niix
#             subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
#             print(f"[成功] {subject_id} - {image_id} 转换完成")
#         except subprocess.CalledProcessError:
#             print(f"[错误] dcm2niix 转换失败: {dcm_folder}")
#     else:
#         # 如果真的找不到 ID，打印出来看看到底是什么奇怪的路径
#         pass 
#         # print(f"[跳过] 路径中缺少受试者ID或图像ID: {sample_file_path}")

# def main():
#     os.makedirs(OUTPUT_DIR, exist_ok=True)
    
#     print("正在扫描目录，寻找最底层数据文件夹...")
#     dcm_folders = set()
    
#     for root, dirs, files in os.walk(INPUT_DIR):
#         # 只要当前目录下有文件（排除全是文件夹的中间层）
#         if files:
#             # 过滤掉元数据文件，看看还有没有剩下的实体数据
#             data_files =[f for f in files if not f.lower().endswith(('.xml', '.csv', '.txt', '.json', '.pdf'))]
#             if data_files:
#                 dcm_folders.add(root)
                
#     dcm_folders = list(dcm_folders)
#     print(f"共发现 {len(dcm_folders)} 个待处理的扫描序列文件夹。")
    
#     if len(dcm_folders) == 0:
#         print("【警告】还是 0 个？请检查：")
#         print("1. /autodl-tmp/ADNI_raw_data 路径是否写错？（试试看进入目录用 ls 命令看一眼）")
#         print("2. 是否解压后多了一层目录？比如 /autodl-tmp/ADNI_raw_data/ADNI")
#         return

#     print(f"开始使用 {NUM_THREADS} 个线程进行批量转换...\n")
    
#     with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
#         executor.map(process_dicom_folder, dcm_folders)
        
#     print("\n全部转换任务执行完毕！")

# if __name__ == "__main__":
#     main()